# TEGR: Bodies of Orbit — Results Viewer

**How to use:**
1. Double-click `LAUNCH_ORBIT.bat` in `Z:\TEGR Collider\` to run the physics on the cluster
2. Wait for the terminal to say `COMPLETE`
3. Come back here and press **Run → Run All Cells** to visualize the results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import json
import glob

# Find the latest SINDy report for the Bodies of Orbit run
sindy_files = sorted(glob.glob('sindy_report_63*'), key=os.path.getmtime, reverse=True)
if sindy_files:
    print(f'Found SINDy report: {sindy_files[0]}')
    with open(sindy_files[0], 'r') as f:
        sindy = json.load(f)
    print('\n=== SINDY EXTRACTION ===')
    for key, val in sindy.items():
        print(f'  {key}: {val}')
else:
    print('No SINDy report found yet. Run LAUNCH_ORBIT.bat first.')

# Load trajectory data
if os.path.exists('aggregate_states.npz'):
    data = np.load('aggregate_states.npz', allow_pickle=True)
    print(f'\nLoaded aggregate_states.npz — keys: {list(data.keys())}')
    if 'states' in data:
        history = data['states']
    elif 'all_states' in data:
        history = data['all_states']
    else:
        first_key = list(data.keys())[0]
        history = data[first_key]
    print(f'Trajectory shape: {history.shape}')
elif os.path.exists('electron_trajectory.npy'):
    history = np.load('electron_trajectory.npy')
    print(f'Loaded electron_trajectory.npy — shape: {history.shape}')
else:
    history = None
    print('No trajectory data found yet. Run LAUNCH_ORBIT.bat first.')

In [ ]:
if history is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # --- Plot 1: Top-Down Orbit ---
    ax = axes[0]
    ax.set_xlim(-40, 40)
    ax.set_ylim(-40, 40)
    ax.set_aspect('equal')
    ax.set_title('Bodies of Orbit: Top-Down View')
    ax.set_xlabel('X (grid units)')
    ax.set_ylabel('Y (grid units)')
    ax.grid(True, alpha=0.3)
    
    # Sun
    ax.plot(history[0, 0, 1], history[0, 0, 2], 'yo', markersize=15, label='Sun', zorder=5)
    # Venus trail
    ax.plot(history[:, 1, 1], history[:, 1, 2], 'r-', alpha=0.4, linewidth=0.5, label='Venus trail')
    # Venus final
    ax.plot(history[-1, 1, 1], history[-1, 1, 2], 'ro', markersize=8, label='Venus (final)', zorder=5)
    ax.legend()
    
    # --- Plot 2: Gamma Over Time ---
    ax2 = axes[1]
    ticks = np.arange(len(history))
    gamma_venus = history[:, 1, 9]
    ax2.plot(ticks, gamma_venus, 'b-', linewidth=0.8)
    ax2.set_title('Venus Gamma Over Time')
    ax2.set_xlabel('Tick')
    ax2.set_ylabel('Gamma')
    ax2.grid(True, alpha=0.3)
    
    # --- Plot 3: Radius Over Time ---
    ax3 = axes[2]
    dx = history[:, 1, 1] - history[:, 0, 1]
    dy = history[:, 1, 2] - history[:, 0, 2]
    r = np.sqrt(dx**2 + dy**2)
    ax3.plot(ticks, r, 'g-', linewidth=0.8)
    ax3.set_title('Orbital Radius Over Time')
    ax3.set_xlabel('Tick')
    ax3.set_ylabel('R (grid units)')
    ax3.axhline(y=20.0, color='r', linestyle='--', alpha=0.5, label='Target R=20')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('bodies_of_orbit_result.png', dpi=150)
    plt.show()
    print('Plot saved to bodies_of_orbit_result.png')
else:
    print('No data to plot.')